In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

# ── Edit this path to wherever you put the zip in your Drive ──
ZIP_PATH = '/content/drive/MyDrive/Datasets/hemo_dataset.zip'
EXTRACT_DIR = '/content/hemo_data'

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)

print("✅ Extracted files:")
for f in os.listdir(EXTRACT_DIR):
    size_mb = os.path.getsize(os.path.join(EXTRACT_DIR, f)) / 1e6
    print(f"   {f}  ({size_mb:.1f} MB)")



In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Verify GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — training will be slow. Check Runtime > Change runtime type.")


In [ ]:
X = np.load(os.path.join(EXTRACT_DIR, 'hemo_X_train.npy'))  # [N, 1, 100]
Y = np.load(os.path.join(EXTRACT_DIR, 'hemo_Y_train.npy'))  # [N]

print(f"X shape : {X.shape}  dtype={X.dtype}")
print(f"Y shape : {Y.shape}  dtype={Y.dtype}")

n_safe  = int((Y == 0).sum())
n_crash = int((Y == 1).sum())
total   = len(Y)
print(f"\nClass balance:")
print(f"  Safe  (0): {n_safe:>7,}  ({n_safe/total*100:.1f}%)")
print(f"  Crash (1): {n_crash:>7,}  ({n_crash/total*100:.1f}%)")

# Quick sanity plot
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
t = np.arange(100) * 0.01

safe_idx  = np.where(Y == 0)[0][0]
crash_idx = np.where(Y == 1)[0][0]

axes[0].plot(t, X[safe_idx, 0, :], color='#2ecc71')
axes[0].set_title('Safe sample')
axes[0].set_xlabel('Time (s)')

axes[1].plot(t, X[crash_idx, 0, :], color='#e74c3c')
axes[1].set_title('Crash-imminent sample')
axes[1].set_xlabel('Time (s)')

plt.suptitle('Hemo-Scout — Sample Waveforms', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:

# ── Hyperparameters (edit freely) ──
BATCH_SIZE    = 64
VAL_SPLIT     = 0.20   # 80/20 train/val
RANDOM_SEED   = 42

torch.manual_seed(RANDOM_SEED)


class HemoDataset(Dataset):
    """Wraps X [N, 1, 100] and Y [N] numpy arrays."""

    def __init__(self, X: np.ndarray, Y: np.ndarray):
        # X is already float32; Y needs to be float for BCEWithLogitsLoss
        self.X = torch.from_numpy(X)               # [N, 1, 100]
        self.Y = torch.from_numpy(Y).float()        # [N]

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]   # returns ([1, 100], scalar)


full_dataset = HemoDataset(X, Y)

val_size   = int(len(full_dataset) * VAL_SPLIT)
train_size = len(full_dataset) - val_size

train_ds, val_ds = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(RANDOM_SEED)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches : {len(train_loader)}  ({train_size:,} samples)")
print(f"Val   batches : {len(val_loader)}  ({val_size:,} samples)")

# Verify a single batch
xb, yb = next(iter(train_loader))
print(f"\nSample batch — X: {xb.shape}  Y: {yb.shape}  Y dtype: {yb.dtype}")



In [ ]:

class HemoScout1DCNN(nn.Module):
    """
    Lightweight 1D-CNN for binary classification of 1-second PLETH windows.

    Input:  [batch, 1, 100]   — 1 channel, 100 time steps
    Output: [batch, 1]        — raw logit (pass through sigmoid for probability)

    Architecture:
      Conv block 1 → 32 filters, kernel 7
      Conv block 2 → 64 filters, kernel 5
      Conv block 3 → 128 filters, kernel 3
      GlobalAvgPool → Dropout → Linear(128 → 1)

    Why GlobalAvgPool instead of Flatten?
      After 3× MaxPool the sequence length is 100→50→25→12 (≈12).
      GlobalAvgPool collapses any remaining length to 1, making the
      classifier head completely size-agnostic and adding implicit
      regularization — important on a small dataset.
    """

    def __init__(self, dropout: float = 0.4):
        super().__init__()

        # ── Feature extractor ──
        self.features = nn.Sequential(

            # Block 1: 1 → 32 channels,  100 → 50 time steps
            nn.Conv1d(in_channels=1,  out_channels=32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),    # 100 → 50

            # Block 2: 32 → 64 channels,  50 → 25 time steps
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),    # 50 → 25

            # Block 3: 64 → 128 channels,  25 → 12 time steps
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),    # 25 → 12
        )

        # ── Classifier head ──
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),        # [batch, 128, 12] → [batch, 128, 1]
            nn.Flatten(),                   # [batch, 128]
            nn.Dropout(dropout),
            nn.Linear(128, 1),             # raw logit
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x.squeeze(1)               # [batch] not [batch, 1]


# Instantiate and inspect
model = HemoScout1DCNN(dropout=0.4).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters  : {total_params:,}")
print(f"Trainable params  : {trainable:,}")

# Dry run to confirm shapes
dummy = torch.zeros(1, 1, 100).to(device)
with torch.no_grad():
    out = model(dummy)
print(f"Output shape      : {out.shape}  (expected: torch.Size([1]))")



In [ ]:

# ── Handle class imbalance ──
# If the dataset is heavily skewed (e.g. 90% safe), BCEWithLogitsLoss
# accepts a pos_weight tensor that upweights the minority (crash) class.
pos_weight_value = n_safe / max(n_crash, 1)
pos_weight = torch.tensor([pos_weight_value], device=device)
print(f"pos_weight for crash class: {pos_weight_value:.2f}x")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# ── Optimizer ──
LR           = 1e-3
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# ── LR scheduler: halve LR if val loss plateaus for 5 epochs ──
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=8, factor=0.5
)


def run_epoch(loader, training: bool):
    """Run one epoch. Returns (avg_loss, accuracy, auc)."""
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    all_logits, all_labels = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for X_batch, Y_batch in loader:
            X_batch = X_batch.to(device, non_blocking=True)
            Y_batch = Y_batch.to(device, non_blocking=True)

            logits = model(X_batch)
            loss   = criterion(logits, Y_batch)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * len(Y_batch)
            preds       = (logits.sigmoid() > 0.5).long()
            correct    += (preds == Y_batch.long()).sum().item()
            n          += len(Y_batch)

            all_logits.append(logits.detach().cpu())
            all_labels.append(Y_batch.cpu())

    avg_loss = total_loss / n
    accuracy = correct / n

    all_logits = torch.cat(all_logits).numpy()
    all_labels = torch.cat(all_labels).numpy()

    # AUC (only meaningful if both classes present in the batch)
    try:
        auc = roc_auc_score(all_labels, all_logits)
    except ValueError:
        auc = float('nan')

    return avg_loss, accuracy, auc



In [ ]:
NUM_EPOCHS    = 40
PATIENCE      = 15      # Early stopping patience (epochs without val improvement)
SAVE_PATH     = '/content/hemo_scout.pth'

best_val_loss = float('inf')
epochs_no_improve = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_auc': []}

print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>9}  {'Val Loss':>8}  {'Val Acc':>7}  {'Val AUC':>7}  {'LR':>8}")
print("─" * 75)

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc, _        = run_epoch(train_loader, training=True)
    vl_loss, vl_acc, vl_auc  = run_epoch(val_loader,   training=False)

    scheduler.step(vl_loss)
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)
    history['val_auc'].append(vl_auc)

    marker = ''
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), SAVE_PATH)
        marker = ' ✅'
    else:
        epochs_no_improve += 1
        marker = f' ({epochs_no_improve}/{PATIENCE})'

    print(f"{epoch:5d}  {tr_loss:10.4f}  {tr_acc:8.2%}  {vl_loss:8.4f}  "
          f"{vl_acc:6.2%}  {vl_auc:7.4f}  {current_lr:.2e}{marker}")

    if epochs_no_improve >= PATIENCE:
        print(f"\n⏹  Early stopping at epoch {epoch}.")
        break

print(f"\n✅ Best validation loss: {best_val_loss:.4f}")
print(f"   Weights saved → {SAVE_PATH}")

In [ ]:
epochs_ran = len(history['train_loss'])
x_axis     = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(x_axis, history['train_loss'], label='Train')
axes[0].plot(x_axis, history['val_loss'],   label='Val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(x_axis, history['train_acc'], label='Train')
axes[1].plot(x_axis, history['val_acc'],   label='Val')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

axes[2].plot(x_axis, history['val_auc'], color='purple', label='Val AUC')
axes[2].set_title('Validation AUC-ROC')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim(0, 1)

plt.suptitle('Hemo-Scout Training Curves', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()
print("Saved training_curves.png")


In [ ]:

# Load best weights and run on full validation set
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
model.eval()

all_logits, all_labels = [], []

with torch.no_grad():
    for X_batch, Y_batch in val_loader:
        X_batch = X_batch.to(device)
        logits  = model(X_batch)
        all_logits.append(logits.cpu())
        all_labels.append(Y_batch)

all_logits = torch.cat(all_logits).numpy()
all_labels = torch.cat(all_labels).numpy()
all_probs  = 1 / (1 + np.exp(-all_logits))   # sigmoid
all_preds  = (all_probs > 0.5).astype(int)

print("── Classification Report (threshold = 0.50) ──")
print(classification_report(all_labels, all_preds, target_names=['Safe', 'Crash']))
print(f"AUC-ROC: {roc_auc_score(all_labels, all_logits):.4f}")


In [ ]:

# File size check
size_bytes = os.path.getsize(SAVE_PATH)
size_kb    = size_bytes / 1024
size_mb    = size_bytes / 1e6
print(f"hemo_scout.pth size: {size_mb:.2f} MB  ({size_kb:.0f} KB)")

if size_mb < 5:
    print("✅ File is under 5 MB — lightweight and edge-ready.")
else:
    print("⚠️  File is over 5 MB — consider reducing filter counts.")

# Verify the state_dict loads and produces output
test_model = HemoScout1DCNN().to(device)
test_model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
test_model.eval()

dummy_input = torch.randn(1, 1, 100).to(device)
with torch.no_grad():
    test_out = test_model(dummy_input).sigmoid().item()
print(f"\nDry-run output (random input): crash probability = {test_out:.4f}")
print("✅ Weights load and run correctly.")

# Copy to Drive for safekeeping
import shutil
drive_save_path = '/content/drive/MyDrive/hemo_scout.pth'
shutil.copy(SAVE_PATH, drive_save_path)
print(f"\n📁 Saved to Drive → {drive_save_path}")
print("\n🏁 CTDE handoff complete. hemo_scout.pth is ready for edge deployment.")
